# Experiment 3: Predict the Best Strategy on a New Task

**Learning objective:** given a *new* task (GSM8K), predict whether search or verification wins under a fixed budget, then verify your prediction by running the experiment.

**The prediction contract (what your predictor must look like):**

```
predict_best_ratio(task_meta) -> "search" | "verify"
    task_meta (dict):
        dataset:         dataset name (e.g., "GSM8K")
        model:           one of the six models (e.g., "qwen3-0.6b")
        p:               single-candidate correctness (from the estimate_p probe below)
        budget:          per-problem budget B (e.g., 50)
        judge_quality:   verifier quality estimate ("weak" | "strong", default "weak")
    returns: the predicted winning direction ("search" = majority vote wins;
             "verify" = score-selection wins)
```

Automatic scoring: your prediction scores 1 point if it matches the measured direction, 0 otherwise.

**Your task:** implement the body of `predict_best_ratio` using the core principles from the earlier notebooks (the simplified model and the decision tree). What determines the winning strategy for a given p, budget and verifier quality?


---

## 1. Implement Your Predictor

Fill in the function below. Hints: follow the decision tree, starting from p (the first decision variable), then budget tier, then verifier quality.


In [ ]:
# --- Your implementation ----------------------------------------------------
def predict_best_ratio(task_meta):
    """Predict the winning strategy direction for a task.

    task_meta: dict with keys dataset / model / p / budget / judge_quality.
    Returns: "search" or "verify".
    """
    # TODO: implement using the decision tree logic (three budget tiers):
    #   1) budget tier: B < 10 | 10 <= B < 100 | B >= 100
    #   2) p: guaranteed zone (p > 0.6), grey zone (0.5 < p < 0.6, decide by
    #      measurement), no-guarantee zone (p <= 0.5, depends on error dispersion)
    #   3) verifier quality: weak (self-scoring) vs. strong (trained PRM)
    # Return None until you implement it - the scorer gives 0 points for None.
    return None   # <-- replace with your reasoning
# ----------------------------------------------------------------------------
print("predict_best_ratio defined (contract signature intact).")

In [ ]:
# --- Configuration (reuse your Notebook 01 model) ---
import os, subprocess, sys, json
# Windows: make CUDA runtime DLLs findable (GPU builds of llama-cpp-python need them)
if os.name == "nt":
    _torch_lib = os.path.join(sys.prefix, "Lib", "site-packages", "torch", "lib")
    if os.path.isdir(_torch_lib):
        os.environ["PATH"] = _torch_lib + os.pathsep + os.environ.get("PATH", "")
        os.add_dll_directory(_torch_lib)
# Dataset mirror (huggingface.co unreachable in some regions)
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

MODEL = "gemma-3-1b"      # reuse your Notebook 01 model
print("Model:", MODEL)

---

## 2. Estimate p with a Small Probe

p (single-candidate correctness) enters the decision tree first. The probe below uses the precomputed MATH-500 pass@1 cache included with the course as a reference (the first 10 problems). Note that this p is from MATH-500, not GSM8K. GSM8K is usually considered the easier task (Notebook 00's self-check question 3), yet in our runs the measured single-candidate accuracy did not follow that pattern. It was lower on GSM8K for Gemma-3-4B (0.54 vs 0.61) but higher for Qwen3-4B (0.78 vs 0.73), both in matched reruns. Run-to-run variability is real: the probe reads only the first 10 problems of an earlier run's cache, so its p is still a noisier estimate, and you should expect it to differ from the numbers above. Possible reasons include zero-shot prompting, sampling temperature, format sensitivity, or subset composition. So do not assume the GSM8K p; measure it. After you run the GSM8K experiment in Section 4, estimate the GSM8K p from your own records and use that estimate when you fill in `p_measured` below. Caveat: GSM8K is part of these models' training corpora, so this is a *dataset-migration* test, not a fully out-of-distribution one. Interpret the migration conclusion accordingly.


In [ ]:
# Estimate p (single-candidate correctness) with a small probe
import os, subprocess, sys, json

# Locate the repository root (walk up until scripts/run_experiment.py is found)
# and switch to it, so relative paths work no matter where Jupyter was started.
_REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_REPO_ROOT, "scripts", "run_experiment.py")):
    _parent = os.path.dirname(_REPO_ROOT)
    if _parent == _REPO_ROOT:
        raise RuntimeError("Could not locate the repository root (scripts/run_experiment.py not found).")
    _REPO_ROOT = _parent
if os.path.abspath(os.getcwd()) != _REPO_ROOT:
    os.chdir(_REPO_ROOT)
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

cmd = [sys.executable, "scripts/predict_best_ratio.py", "--model", MODEL, "--n", "10"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Probe done - note the reported p.")

---

## 3. Record Your Prediction

Fill in the p you just measured, then call your predictor and record its output. For now, fill `p_measured` with the MATH-500 probe value from Section 2. Your model's GSM8K p may differ; after Section 4, update `p_measured` with your own GSM8K estimate, re-run this cell, and then run the scoring cell at the end of Section 4 to see your new score. To estimate it, take the first generated answer for each of your 6 problems and judge those answers by hand; records are in `data/results/gsm8k_<model>/records.jsonl`.


In [ ]:
# Record your measured p and make the prediction
p_measured = 0.30        # <-- start with your Section-2 MATH-500 probe value; after Section 4
                         #     you can update it with your GSM8K estimate and re-run
task_meta = {
    "dataset": "GSM8K",
    "model": MODEL,
    "p": p_measured,
    "budget": 50,
    "judge_quality": "weak",
}
prediction = predict_best_ratio(task_meta)
print("Your prediction:", prediction)
print("Rationale: write 2-3 sentences explaining which decision-tree branch you followed.")

---

## 4. Run the Experiment and Score Your Prediction

Run the GSM8K headline configs (N=50/M=0, N=25/M=1, N=5/M=9; the same three configs as the 3-core set in Notebook 01) and compare with your prediction.

> **First run downloads the dataset** (GSM8K, ≈3 MB) from the Hugging Face Hub; later runs reuse the local cache. The setup cell above already tries the mirror automatically; if a download still fails, see the mirror workaround in Notebook 01.
>
> **6-problem noise note:** with only 6 problems, accuracy estimates carry roughly ±20 percentage points at one standard error. The measured *direction* may flip purely from noise. Treat the score as a sanity check on your reasoning, not a precise measurement.
>
> <details><summary>Our GSM8K numbers (check after you run your own experiment)</summary>
>
> On our 100-problem seed-0 GSM8K subset the majority-vote accuracy was *lower* than on MATH for both models (Gemma-3-4B: 0.59 vs 0.73; Qwen3-4B: 0.81 vs 0.86), and the single-candidate p moved in different directions (Section 2): lower for Gemma-3-4B, higher for Qwen3-4B. The "easier task means a higher p" intuition did **not** hold consistently in our zero-shot setup. If your prediction assumed it did, that is itself a finding worth discussing: task "difficulty" interacts with prompting and format sensitivity, so p must be measured, not assumed.
>
> </details>


In [ ]:
# Run the GSM8K experiment (headline three configs, 1 seed, 6 problems)
# Expected runtime: a few hours on a laptop GPU with a small model. The run is
# incremental - you can interrupt it and resume with the same command.
cmd = [sys.executable, "scripts/run_experiment.py",
       "--model", MODEL, "--seeds", "0", "--dataset", "GSM8K",
       "--configs", "N=50,M=0", "N=25,M=1", "N=5,M=9",
       "--max-problems", "6",
       "--out-dir", "data/results/gsm8k_" + MODEL, "--finalize"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("GSM8K experiment finished.")

In [ ]:
# Automatic scoring: compare prediction with the measured direction (tolerant)
import json, os

exp_dir = os.path.join("data", "results", "gsm8k_" + MODEL)
exp_file = os.path.join(exp_dir, "exp1.json")
if os.path.isdir(exp_dir):
    try:
        from scripts.analyze_experiment import main as analyze_main
        analyze_main(["--records", exp_dir, "--model", MODEL,
                      "--dataset", "GSM8K", "--out", exp_file])
    except Exception as e:
        if not os.listdir(exp_dir):
            print("Note: no GSM8K data yet - run the experiment cell first, then re-run this cell.")
        else:
            print("Aggregation failed:", type(e).__name__, str(e)[:200])
configs = None
source = "your run"
if os.path.exists(exp_file):
    with open(exp_file, encoding="utf-8") as f:
        exp = json.load(f)
    configs = exp["configs"]
else:
    # Cache fallback: use the included GSM8K headline data (data/cache_subset/)
    hl_path = os.path.join("data", "cache_subset", "headline.json")
    if os.path.exists(hl_path):
        for h in json.load(open(hl_path, encoding="utf-8"))["headline"]:
            if h.get("key") == MODEL and h["dataset"] == "GSM8K":
                configs = [rec for rec in h["headline"].values() if rec]
                source = "the included cache (our 100-problem GSM8K run, seed 0)"
                break
        if configs is None:
            # Your model has no cached GSM8K data (we ran GSM8K on gemma-3-4b and
            # qwen3-4b only) - use qwen3-4b's numbers as a reference for scoring.
            for h in json.load(open(hl_path, encoding="utf-8"))["headline"]:
                if h.get("key") == "qwen3-4b" and h["dataset"] == "GSM8K":
                    configs = [rec for rec in h["headline"].values() if rec]
                    source = "the included cache (qwen3-4b reference - your model has no cached GSM8K run)"
                    break
    else:
        print("Note: no local GSM8K data and no cache - run the experiment cell first.")
if configs:
    from scripts.predict_best_ratio import actual_direction_from_configs, score_prediction
    actual = actual_direction_from_configs(configs)
    if prediction is None:
        print("Your predictor is not implemented yet (returns None) - implement it in Section 1, then re-run.")
        score = 0
    else:
        score = score_prediction(prediction, actual)
        print("Measured direction (%s):" % source, actual, "| Your prediction:", prediction, "| Score:", score, "/ 1")
    for c in configs:
        print("  N=%d M=%d accuracy=%.3f" % (c["config"]["N"], c["config"]["M"], c["accuracy"]))

---

## 5. Analysis Questions (Answer in your own words)

1. **Did your prediction match the measurement?** If yes, which decision-tree branch did the data confirm? If no, which assumption was wrong: the p estimate, the verifier quality, or the budget tier?

2. **Difficulty migration:** GSM8K is easier than MATH (Notebook 00's self-check question 3). Did the strategy outcome change on the easier task? (Estimate the GSM8K p from your Section 4 records and compare it with the MATH-500 p the probe reported.)

3. **Generalization:** would your predictor logic transfer to a task you have never run (e.g., a code-generation benchmark)? What additional information about p, budget and verifier quality would you need, and what would the model fail to capture (task-specific factors)?


---

## Summary

- You implemented a strategy predictor from the core principles, probed p on a new task, and scored your prediction against the measurement.
- If your prediction was wrong, that is where the learning is: find the branch that failed.

Proceed to `04_baseline_review.ipynb` (advanced) when ready.
